workspace.bronze.trichy_east_boothwise_votes_raw

In [0]:

# Databricks notebook source

# Python utilities for JSON handling, timestamps and unique batch IDs.
import json
from datetime import datetime, timezone

# Spark functions and data types used to build the Bronze dataframe.
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)


# Project and Unity Catalog configuration.
PROJECT_NAME = "trichy_east_election_pipeline"
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"


# Root Databricks Volume containing the year folders.
BASE_VOLUME_PATH = "/Volumes/workspace/elections/trichy_east"


# Stable constituency metadata added even when a PDF header is unreadable.
CONSTITUENCY_NO = "141"
CONSTITUENCY_NAME = "Tiruchirappalli East"


# Only Form 20 boothwise-vote sources are processed by this notebook.
SOURCE_TYPE = "boothwise_votes"


# Unique identifier for this extraction run.
# This lets every Bronze row be traced back to one notebook execution.
BATCH_ID = datetime.now(timezone.utc).strftime(
    "boothwise_%Y%m%dT%H%M%SZ"
)


# Raw Bronze target table.
BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.trichy_east_boothwise_votes_raw"
)


print("Project:", PROJECT_NAME)
print("Source type:", SOURCE_TYPE)
print("Batch ID:", BATCH_ID)
print("Target table:", BRONZE_TABLE)

In [0]:
# Standard Python libraries used to locate and read the JSON manifest.
import json
from pathlib import Path


# The notebook currently sits at the Git repository root.
MANIFEST_PATH = Path("config/source_manifest.json")


# Stop early with a clear error if Git has not synced the configuration.
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Source manifest was not found at {MANIFEST_PATH}. "
        "Pull the latest Git changes in Databricks."
    )


# Load the configuration maintained in GitHub.
with MANIFEST_PATH.open("r") as manifest_file:
    source_manifest = json.load(manifest_file)


# Confirm that Git configuration points to the intended Databricks Volume.
assert (
    source_manifest["databricks_base_volume"]
    == BASE_VOLUME_PATH
), "The notebook and manifest Volume paths do not match."


# Select only boothwise Form 20 sources.
boothwise_source_rows = []

for source in source_manifest["sources"]:
    if source["source_type"] != SOURCE_TYPE:
        continue

    source_file_path = (
        f"{BASE_VOLUME_PATH.rstrip('/')}/"
        f"{source['relative_path'].lstrip('/')}"
    )

    boothwise_source_rows.append(
        {
            "project_name": PROJECT_NAME,
            "constituency_no": CONSTITUENCY_NO,
            "constituency_name": CONSTITUENCY_NAME,
            "election_year": int(source["election_year"]),
            "source_type": source["source_type"],
            "source_format": source["format"],
            "source_file_name": source["file_name"],
            "source_file_path": source_file_path,
            "configured_extraction_strategy": source[
                "extraction_strategy"
            ],
            "batch_id": BATCH_ID,
        }
    )


# Convert the three selected configurations into a Spark dataframe.
boothwise_source_df = spark.createDataFrame(
    boothwise_source_rows
)


display(
    boothwise_source_df.orderBy("election_year")
)


print(
    "Configured boothwise files:",
    boothwise_source_df.count()
)

In [0]:
# Standard Python libraries used to locate and read the JSON manifest.
import json
from pathlib import Path


# The notebook currently sits at the Git repository root.
MANIFEST_PATH = Path("config/source_manifest.json")


# Stop early with a clear error if Git has not synced the configuration.
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Source manifest was not found at {MANIFEST_PATH}. "
        "Pull the latest Git changes in Databricks."
    )


# Load the configuration maintained in GitHub.
with MANIFEST_PATH.open("r") as manifest_file:
    source_manifest = json.load(manifest_file)


# Confirm that Git configuration points to the intended Databricks Volume.
assert (
    source_manifest["databricks_base_volume"]
    == BASE_VOLUME_PATH
), "The notebook and manifest Volume paths do not match."


# Select only boothwise Form 20 sources.
boothwise_source_rows = []

for source in source_manifest["sources"]:
    if source["source_type"] != SOURCE_TYPE:
        continue

    source_file_path = (
        f"{BASE_VOLUME_PATH.rstrip('/')}/"
        f"{source['relative_path'].lstrip('/')}"
    )

    boothwise_source_rows.append(
        {
            "project_name": PROJECT_NAME,
            "constituency_no": CONSTITUENCY_NO,
            "constituency_name": CONSTITUENCY_NAME,
            "election_year": int(source["election_year"]),
            "source_type": source["source_type"],
            "source_format": source["format"],
            "source_file_name": source["file_name"],
            "source_file_path": source_file_path,
            "configured_extraction_strategy": source[
                "extraction_strategy"
            ],
            "batch_id": BATCH_ID,
        }
    )


# Convert the three selected configurations into a Spark dataframe.
boothwise_source_df = spark.createDataFrame(
    boothwise_source_rows
)


display(
    boothwise_source_df.orderBy("election_year")
)


print(
    "Configured boothwise files:",
    boothwise_source_df.count()
)

In [0]:
# Python utility used to check file paths exposed through the Databricks Volume.
import os


# Preserve one availability result for each Form 20 file.
boothwise_file_check_rows = []

for source in boothwise_source_rows:
    source_file_path = source["source_file_path"]

    try:
        file_exists = os.path.isfile(source_file_path)

        if file_exists:
            file_size_bytes = int(
                os.path.getsize(source_file_path)
            )
            file_status = "AVAILABLE"
            validation_error = None
        else:
            file_size_bytes = None
            file_status = "MISSING"
            validation_error = (
                "Configured Form 20 file was not found."
            )

    except Exception as error:
        file_exists = False
        file_size_bytes = None
        file_status = "ERROR"
        validation_error = str(error)

    boothwise_file_check_rows.append(
        {
            **source,
            "file_exists": file_exists,
            "file_size_bytes": file_size_bytes,
            "file_status": file_status,
            "validation_error": validation_error,
        }
    )


# Define explicit types because validation_error may be null for every file.
boothwise_file_check_schema = StructType([
    StructField("project_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("constituency_name", StringType(), False),
    StructField("election_year", IntegerType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_format", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField(
        "configured_extraction_strategy",
        StringType(),
        False
    ),
    StructField("batch_id", StringType(), False),
    StructField("file_exists", StringType(), False),
    StructField("file_size_bytes", StringType(), True),
    StructField("file_status", StringType(), False),
    StructField("validation_error", StringType(), True),
])


# Convert validation results into a Spark dataframe.
boothwise_file_check_df = spark.createDataFrame(
    boothwise_file_check_rows,
    schema=boothwise_file_check_schema
)


display(
    boothwise_file_check_df.select(
        "election_year",
        "source_file_name",
        "source_file_path",
        "file_size_bytes",
        "file_status",
        "validation_error"
    ).orderBy("election_year")
)


unavailable_file_count = (
    boothwise_file_check_df
    .filter(F.col("file_status") != "AVAILABLE")
    .count()
)


print("Form 20 files:", boothwise_file_check_df.count())
print("Unavailable files:", unavailable_file_count)


if unavailable_file_count > 0:
    raise ValueError(
        f"{unavailable_file_count} Form 20 file(s) are unavailable."
    )

In [0]:
# Install the PDF reader used to extract text page by page.
# Pinning the version makes future notebook runs more reproducible.
%pip install pypdf==6.10.0

In [0]:
# Import the PDF reader after installation.
from pypdf import PdfReader

# Print the installed version as evidence of the extraction environment.
import pypdf

print("pypdf version:", pypdf.__version__)
print("PDF extraction dependency: READY")

In [0]:
# Define every Bronze column explicitly.
# Explicit schemas prevent Spark from guessing different types across years.
BRONZE_SCHEMA_DEFINITION = StructType([
    # Source lineage
    StructField("project_name", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField("source_type", StringType(), False),

    # Stable election metadata
    StructField("constituency_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("election_year", IntegerType(), False),

    # Position inside the source PDF
    StructField("page_no", IntegerType(), True),
    StructField("table_no", IntegerType(), True),
    StructField("row_no", IntegerType(), True),

    # Raw booth identifiers when they can be safely observed
    StructField("polling_station_no_raw", StringType(), True),
    StructField("polling_station_name_raw", StringType(), True),

    # Complete extracted row text
    StructField("raw_row_text", StringType(), True),

    # Variable source columns preserved as JSON strings
    StructField("raw_columns_json", StringType(), True),
    StructField("candidate_columns_json", StringType(), True),
    StructField("totals_json", StringType(), True),

    # Extraction diagnostics preserved as JSON
    StructField(
        "extraction_metadata_json",
        StringType(),
        True
    ),

    # Parsing outcome
    StructField("parse_status", StringType(), False),
    StructField("parse_error", StringType(), True),
    StructField("extraction_method", StringType(), False),

    # Run lineage
    StructField("batch_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("extraction_timestamp", TimestampType(), True),
])


print(
    "Bronze schema columns:",
    len(BRONZE_SCHEMA_DEFINITION.fields)
)

print(
    "Target table:",
    BRONZE_TABLE
)

In [0]:
def clean_extracted_line(raw_line):
    """
    Apply only technical cleanup needed to store readable text.

    This intentionally avoids election business rules or column mapping.
    """
    if raw_line is None:
        return ""

    # Remove null characters that can appear in PDF extraction.
    cleaned_line = raw_line.replace("\x00", "")

    # Normalize repeated whitespace while preserving the text itself.
    cleaned_line = " ".join(cleaned_line.split())

    return cleaned_line.strip()


def extract_pdf_pages(source):
    """
    Extract every page and return raw Bronze-compatible records.

    Successful pages create one record per non-empty extracted line.
    Failed or empty pages still create one FAILED record so that source
    coverage remains visible.
    """
    extracted_records = []
    extraction_timestamp = datetime.now(timezone.utc)

    try:
        pdf_reader = PdfReader(source["source_file_path"])

        total_pdf_pages = len(pdf_reader.pages)

        for page_index, pdf_page in enumerate(
            pdf_reader.pages,
            start=1
        ):
            try:
                page_text = pdf_page.extract_text(
                    extraction_mode="layout"
                ) or ""

                extracted_lines = [
                    clean_extracted_line(line)
                    for line in page_text.splitlines()
                ]

                # Empty lines provide no usable evidence and are skipped.
                extracted_lines = [
                    line
                    for line in extracted_lines
                    if line
                ]

                if not extracted_lines:
                    extracted_records.append(
                        {
                            "project_name": source["project_name"],
                            "source_file_name": source["source_file_name"],
                            "source_file_path": source["source_file_path"],
                            "source_type": source["source_type"],
                            "constituency_name": source["constituency_name"],
                            "constituency_no": source["constituency_no"],
                            "election_year": source["election_year"],
                            "page_no": page_index,
                            "table_no": None,
                            "row_no": None,
                            "polling_station_no_raw": None,
                            "polling_station_name_raw": None,
                            "raw_row_text": None,
                            "raw_columns_json": None,
                            "candidate_columns_json": None,
                            "totals_json": None,
                            "extraction_metadata_json": json.dumps({
                                "pdf_page": page_index,
                                "pdf_page_count": total_pdf_pages,
                                "extracted_line_count": 0,
                                "parser": "pypdf",
                            }),
                            "parse_status": "FAILED",
                            "parse_error": (
                                "No text extracted from PDF page."
                            ),
                            "extraction_method": (
                                "pypdf_layout_text"
                            ),
                            "batch_id": source["batch_id"],
                            "ingestion_timestamp": None,
                            "extraction_timestamp": extraction_timestamp,
                        }
                    )

                    continue

                for row_index, extracted_line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    extracted_records.append(
                        {
                            "project_name": source["project_name"],
                            "source_file_name": source["source_file_name"],
                            "source_file_path": source["source_file_path"],
                            "source_type": source["source_type"],
                            "constituency_name": source["constituency_name"],
                            "constituency_no": source["constituency_no"],
                            "election_year": source["election_year"],
                            "page_no": page_index,
                            "table_no": 1,
                            "row_no": row_index,
                            "polling_station_no_raw": None,
                            "polling_station_name_raw": None,
                            "raw_row_text": extracted_line,

                            # Every extracted line is preserved without
                            # assuming what its columns mean.
                            "raw_columns_json": json.dumps({
                                "line_1": extracted_line
                            }),

                            # Candidate and total mappings are intentionally
                            # deferred to Silver.
                            "candidate_columns_json": None,
                            "totals_json": None,

                            "extraction_metadata_json": json.dumps({
                                "pdf_page": page_index,
                                "pdf_page_count": total_pdf_pages,
                                "extracted_line_count": len(
                                    extracted_lines
                                ),
                                "parser": "pypdf",
                                "extraction_mode": "layout",
                            }),
                            "parse_status": "SUCCESS",
                            "parse_error": None,
                            "extraction_method": (
                                "pypdf_layout_text"
                            ),
                            "batch_id": source["batch_id"],
                            "ingestion_timestamp": None,
                            "extraction_timestamp": extraction_timestamp,
                        }
                    )

            except Exception as page_error:
                # A page-level error must not prevent other pages or files
                # from being processed.
                extracted_records.append(
                    {
                        "project_name": source["project_name"],
                        "source_file_name": source["source_file_name"],
                        "source_file_path": source["source_file_path"],
                        "source_type": source["source_type"],
                        "constituency_name": source["constituency_name"],
                        "constituency_no": source["constituency_no"],
                        "election_year": source["election_year"],
                        "page_no": page_index,
                        "table_no": None,
                        "row_no": None,
                        "polling_station_no_raw": None,
                        "polling_station_name_raw": None,
                        "raw_row_text": None,
                        "raw_columns_json": None,
                        "candidate_columns_json": None,
                        "totals_json": None,
                        "extraction_metadata_json": json.dumps({
                            "pdf_page": page_index,
                            "parser": "pypdf",
                        }),
                        "parse_status": "FAILED",
                        "parse_error": str(page_error),
                        "extraction_method": (
                            "pypdf_layout_text"
                        ),
                        "batch_id": source["batch_id"],
                        "ingestion_timestamp": None,
                        "extraction_timestamp": extraction_timestamp,
                    }
                )

    except Exception as file_error:
        # File-level failures are also retained as Bronze records.
        extracted_records.append(
            {
                "project_name": source["project_name"],
                "source_file_name": source["source_file_name"],
                "source_file_path": source["source_file_path"],
                "source_type": source["source_type"],
                "constituency_name": source["constituency_name"],
                "constituency_no": source["constituency_no"],
                "election_year": source["election_year"],
                "page_no": None,
                "table_no": None,
                "row_no": None,
                "polling_station_no_raw": None,
                "polling_station_name_raw": None,
                "raw_row_text": None,
                "raw_columns_json": None,
                "candidate_columns_json": None,
                "totals_json": None,
                "extraction_metadata_json": json.dumps({
                    "parser": "pypdf"
                }),
                "parse_status": "FAILED",
                "parse_error": str(file_error),
                "extraction_method": "pypdf_layout_text",
                "batch_id": source["batch_id"],
                "ingestion_timestamp": None,
                "extraction_timestamp": extraction_timestamp,
            }
        )

    return extracted_records


print("Raw page extractor: READY")

In [0]:
# Select the earliest election year for a controlled extraction test.
test_source = sorted(
    boothwise_source_rows,
    key=lambda source: source["election_year"]
)[0]


print("Testing file:", test_source["source_file_name"])
print("Election year:", test_source["election_year"])


# Extract the selected PDF using the function defined in Cell 6.
test_extracted_records = extract_pdf_pages(test_source)


# Convert the extracted Python records into a Spark dataframe
# using the stable Bronze schema.
test_bronze_df = spark.createDataFrame(
    test_extracted_records,
    schema=BRONZE_SCHEMA_DEFINITION
)


# Add the actual Spark ingestion timestamp.
test_bronze_df = test_bronze_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


# Summarize the extraction result without displaying every raw row.
display(
    test_bronze_df
    .groupBy(
        "source_file_name",
        "election_year",
        "parse_status",
        "extraction_method"
    )
    .count()
)


# Display a small sample so we can visually inspect the extracted text.
display(
    test_bronze_df.select(
        "page_no",
        "row_no",
        "raw_row_text",
        "parse_status"
    )
    .orderBy("page_no", "row_no")
    .limit(20)
)


print("Test records extracted:", test_bronze_df.count())

In [0]:
# Store extracted records from every configured Form 20 file.
all_boothwise_records = []


# Process files sequentially.
# There are only three source PDFs, so driver-side extraction is simpler,
# easier to debug and appropriate for this portfolio-sized dataset.
for source in sorted(
    boothwise_source_rows,
    key=lambda row: row["election_year"]
):
    print(
        f"Extracting {source['election_year']}: "
        f"{source['source_file_name']}"
    )

    source_records = extract_pdf_pages(source)

    print(
        "Records extracted:",
        len(source_records)
    )

    all_boothwise_records.extend(source_records)


# Build the complete Bronze dataframe using the stable schema.
boothwise_bronze_df = spark.createDataFrame(
    all_boothwise_records,
    schema=BRONZE_SCHEMA_DEFINITION
)


# Record when Spark ingested the extracted records.
boothwise_bronze_df = boothwise_bronze_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


# Summarize extraction results by file and status.
display(
    boothwise_bronze_df
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy(
        "election_year",
        "source_file_name",
        "parse_status"
    )
)


print(
    "Total Bronze records:",
    boothwise_bronze_df.count()
)

In [0]:
# Review failed pages before selecting a fallback extractor.
display(
    boothwise_bronze_df
    .filter(F.col("parse_status") == "FAILED")
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_error"
    )
    .count()
    .orderBy("election_year", "parse_error")
)

In [0]:
# Install PyMuPDF as the fallback for PDFs whose rotated text
# cannot be extracted reliably by pypdf.
%pip install PyMuPDF==1.26.7

In [0]:
# PyMuPDF is imported as fitz.
# It will be used only when the primary pypdf extraction returns no text.
import fitz

print("PyMuPDF version:", fitz.VersionBind)
print("Fallback PDF extractor: READY")

In [0]:
def extract_pdf_pages_with_pymupdf(source):
    """
    Extract raw page text using PyMuPDF.

    This fallback is used only when the primary pypdf extraction fails.
    It preserves the same Bronze schema and still avoids candidate mapping.
    """
    extracted_records = []
    extraction_timestamp = datetime.now(timezone.utc)

    try:
        pdf_document = fitz.open(
            source["source_file_path"]
        )

        total_pdf_pages = pdf_document.page_count

        for page_index in range(total_pdf_pages):
            page_no = page_index + 1

            try:
                pdf_page = pdf_document.load_page(
                    page_index
                )

                # "text" mode works better for some rotated Form 20 PDFs.
                page_text = pdf_page.get_text("text") or ""

                extracted_lines = [
                    clean_extracted_line(line)
                    for line in page_text.splitlines()
                ]

                extracted_lines = [
                    line
                    for line in extracted_lines
                    if line
                ]

                if not extracted_lines:
                    extracted_records.append(
                        {
                            "project_name": source["project_name"],
                            "source_file_name": source["source_file_name"],
                            "source_file_path": source["source_file_path"],
                            "source_type": source["source_type"],
                            "constituency_name": source["constituency_name"],
                            "constituency_no": source["constituency_no"],
                            "election_year": source["election_year"],
                            "page_no": page_no,
                            "table_no": None,
                            "row_no": None,
                            "polling_station_no_raw": None,
                            "polling_station_name_raw": None,
                            "raw_row_text": None,
                            "raw_columns_json": None,
                            "candidate_columns_json": None,
                            "totals_json": None,
                            "extraction_metadata_json": json.dumps({
                                "pdf_page": page_no,
                                "pdf_page_count": total_pdf_pages,
                                "extracted_line_count": 0,
                                "parser": "pymupdf",
                            }),
                            "parse_status": "FAILED",
                            "parse_error": (
                                "No text extracted using PyMuPDF."
                            ),
                            "extraction_method": "pymupdf_text",
                            "batch_id": source["batch_id"],
                            "ingestion_timestamp": None,
                            "extraction_timestamp": extraction_timestamp,
                        }
                    )

                    continue

                for row_index, extracted_line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    extracted_records.append(
                        {
                            "project_name": source["project_name"],
                            "source_file_name": source["source_file_name"],
                            "source_file_path": source["source_file_path"],
                            "source_type": source["source_type"],
                            "constituency_name": source["constituency_name"],
                            "constituency_no": source["constituency_no"],
                            "election_year": source["election_year"],
                            "page_no": page_no,
                            "table_no": 1,
                            "row_no": row_index,
                            "polling_station_no_raw": None,
                            "polling_station_name_raw": None,
                            "raw_row_text": extracted_line,
                            "raw_columns_json": json.dumps({
                                "line_1": extracted_line
                            }),
                            "candidate_columns_json": None,
                            "totals_json": None,
                            "extraction_metadata_json": json.dumps({
                                "pdf_page": page_no,
                                "pdf_page_count": total_pdf_pages,
                                "extracted_line_count": len(
                                    extracted_lines
                                ),
                                "parser": "pymupdf",
                                "extraction_mode": "text",
                                "fallback_reason": (
                                    "Primary pypdf extraction failed."
                                ),
                            }),
                            "parse_status": "SUCCESS",
                            "parse_error": None,
                            "extraction_method": "pymupdf_text",
                            "batch_id": source["batch_id"],
                            "ingestion_timestamp": None,
                            "extraction_timestamp": extraction_timestamp,
                        }
                    )

            except Exception as page_error:
                extracted_records.append(
                    {
                        "project_name": source["project_name"],
                        "source_file_name": source["source_file_name"],
                        "source_file_path": source["source_file_path"],
                        "source_type": source["source_type"],
                        "constituency_name": source["constituency_name"],
                        "constituency_no": source["constituency_no"],
                        "election_year": source["election_year"],
                        "page_no": page_no,
                        "table_no": None,
                        "row_no": None,
                        "polling_station_no_raw": None,
                        "polling_station_name_raw": None,
                        "raw_row_text": None,
                        "raw_columns_json": None,
                        "candidate_columns_json": None,
                        "totals_json": None,
                        "extraction_metadata_json": json.dumps({
                            "pdf_page": page_no,
                            "parser": "pymupdf",
                        }),
                        "parse_status": "FAILED",
                        "parse_error": str(page_error),
                        "extraction_method": "pymupdf_text",
                        "batch_id": source["batch_id"],
                        "ingestion_timestamp": None,
                        "extraction_timestamp": extraction_timestamp,
                    }
                )

        pdf_document.close()

    except Exception as file_error:
        extracted_records.append(
            {
                "project_name": source["project_name"],
                "source_file_name": source["source_file_name"],
                "source_file_path": source["source_file_path"],
                "source_type": source["source_type"],
                "constituency_name": source["constituency_name"],
                "constituency_no": source["constituency_no"],
                "election_year": source["election_year"],
                "page_no": None,
                "table_no": None,
                "row_no": None,
                "polling_station_no_raw": None,
                "polling_station_name_raw": None,
                "raw_row_text": None,
                "raw_columns_json": None,
                "candidate_columns_json": None,
                "totals_json": None,
                "extraction_metadata_json": json.dumps({
                    "parser": "pymupdf"
                }),
                "parse_status": "FAILED",
                "parse_error": str(file_error),
                "extraction_method": "pymupdf_text",
                "batch_id": source["batch_id"],
                "ingestion_timestamp": None,
                "extraction_timestamp": extraction_timestamp,
            }
        )

    return extracted_records


print("PyMuPDF fallback extractor: READY")

In [0]:
# Identify files with no successful primary extraction records.
primary_file_status_df = (
    boothwise_bronze_df
    .groupBy(
        "election_year",
        "source_file_name"
    )
    .agg(
        F.sum(
            F.when(
                F.col("parse_status") == "SUCCESS",
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias("successful_record_count")
    )
)


failed_primary_files = [
    row["source_file_name"]
    for row in (
        primary_file_status_df
        .filter(F.col("successful_record_count") == 0)
        .select("source_file_name")
        .collect()
    )
]


print(
    "Files requiring fallback:",
    failed_primary_files
)


# Keep successful primary records and preserve any partial failures
# for files that produced at least some usable content.
successful_primary_records = [
    record
    for record in all_boothwise_records
    if record["source_file_name"] not in failed_primary_files
]


fallback_records = []


for source in boothwise_source_rows:
    if source["source_file_name"] not in failed_primary_files:
        continue

    print(
        "Fallback extraction:",
        source["source_file_name"]
    )

    extracted_fallback_records = (
        extract_pdf_pages_with_pymupdf(source)
    )

    print(
        "Fallback records:",
        len(extracted_fallback_records)
    )

    fallback_records.extend(
        extracted_fallback_records
    )


# Replace fully failed primary-file records with fallback output.
final_boothwise_records = (
    successful_primary_records
    + fallback_records
)


final_boothwise_bronze_df = spark.createDataFrame(
    final_boothwise_records,
    schema=BRONZE_SCHEMA_DEFINITION
).withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


display(
    final_boothwise_bronze_df
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy(
        "election_year",
        "source_file_name",
        "parse_status"
    )
)


print(
    "Final Bronze records:",
    final_boothwise_bronze_df.count()
)

In [0]:
# Confirm that all three election years produced Bronze records.
source_coverage_df = (
    final_boothwise_bronze_df
    .groupBy(
        "election_year",
        "source_file_name"
    )
    .agg(
        F.count("*").alias("record_count"),
        F.sum(
            F.when(
                F.col("parse_status") == "SUCCESS",
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias("successful_record_count"),
        F.sum(
            F.when(
                F.col("parse_status") == "FAILED",
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias("failed_record_count"),
        F.countDistinct("page_no").alias(
            "observed_page_count"
        )
    )
    .orderBy("election_year")
)


display(source_coverage_df)


# A raw row should be unique within its source file and page.
duplicate_raw_coordinates_df = (
    final_boothwise_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .groupBy(
        "source_file_path",
        "page_no",
        "table_no",
        "row_no"
    )
    .agg(
        F.count("*").alias("duplicate_count")
    )
    .filter(F.col("duplicate_count") > 1)
)


display(duplicate_raw_coordinates_df)


source_count = source_coverage_df.count()

source_without_success_count = (
    source_coverage_df
    .filter(F.col("successful_record_count") == 0)
    .count()
)

duplicate_coordinate_count = (
    duplicate_raw_coordinates_df.count()
)


print("Source files represented:", source_count)
print(
    "Sources without successful records:",
    source_without_success_count
)
print(
    "Duplicate raw coordinates:",
    duplicate_coordinate_count
)


if source_count != 3:
    raise ValueError(
        f"Expected 3 Form 20 source files, found {source_count}."
    )

if source_without_success_count > 0:
    raise ValueError(
        "At least one Form 20 file has no successful records."
    )

if duplicate_coordinate_count > 0:
    raise ValueError(
        "Duplicate raw source coordinates were found."
    )


print("Pre-write Bronze validation: PASS")

In [0]:
# Ensure the Bronze schema exists.
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")


# Create the target table from the stable Bronze schema if needed.
# Writing an empty dataframe establishes the schema without adding records.
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.createDataFrame(
            [],
            schema=BRONZE_SCHEMA_DEFINITION
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE)
    )


# Delete this batch before inserting it.
# This makes rerunning the write cell safe for the same BATCH_ID.
spark.sql(f"""
DELETE FROM {BRONZE_TABLE}
WHERE batch_id = '{BATCH_ID}'
""")


# Append the fully validated records.
(
    final_boothwise_bronze_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE)
)


print(
    "Bronze rows written:",
    final_boothwise_bronze_df.count()
)

print("Batch ID:", BATCH_ID)
print("Bronze table:", BRONZE_TABLE)

In [0]:
# Read this batch back from Delta.
written_boothwise_batch_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)


# Show stored extraction coverage by year and method.
display(
    written_boothwise_batch_df
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_status",
        "extraction_method"
    )
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("page_no").alias(
            "page_count"
        )
    )
    .orderBy(
        "election_year",
        "source_file_name",
        "parse_status"
    )
)


written_record_count = (
    written_boothwise_batch_df.count()
)

expected_record_count = (
    final_boothwise_bronze_df.count()
)

failed_record_count = (
    written_boothwise_batch_df
    .filter(F.col("parse_status") == "FAILED")
    .count()
)

distinct_year_count = (
    written_boothwise_batch_df
    .select("election_year")
    .distinct()
    .count()
)


print("Expected records:", expected_record_count)
print("Written records:", written_record_count)
print("Election years represented:", distinct_year_count)
print("Failed records retained:", failed_record_count)


if written_record_count != expected_record_count:
    raise ValueError(
        "Written Bronze row count does not match the validated dataframe."
    )

if distinct_year_count != 3:
    raise ValueError(
        "The written batch does not contain all three election years."
    )


print("Bronze boothwise validation: PASS")